In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_order_items = spark.read.table('global_partner_project.silver.order_items')

# Creating Dimension Tables

## dim_user table

In [0]:
df_dim_user = df_order_items.select('user_id','is_loyalty').distinct()
df_dim_user.display()
print(df_dim_user.count()) # 22959 rows

In [0]:
agg_df = df_dim_user.groupBy('user_id').agg(F.count('user_id').alias('cnt')).orderBy(F.col('cnt').desc())
agg_df.display()

In [0]:
df_dim_user.filter(F.col('user_id')=='6282877c7603690abc211d85').orderBy('creation_time_utc').display()

In [0]:
df_order_items.filter(F.col('user_id')=='6282877c7603690abc211d85').orderBy('creation_time_utc').display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_user;

In [0]:
df_dim_user.write.mode('append').saveAsTable('global_partner_project.gold.dim_user')

## DIM_USER_SCD2

- Create a lag of previous values
- filter for null values & curr_val != prev_val
- The above 2 steps you would be doing in order to eliminate the repeating values w.r.t partition
- Create a lead of timestamp -> this would be effective_to
- Your cur_timestamp would be effective_from


In [0]:
window = Window.partitionBy('user_id').orderBy('creation_time_utc')

In [0]:
df_user_loyalty = df_order_items.select('creation_time_utc','user_id','is_loyalty').distinct()

In [0]:
df_user_loyalty = df_user_loyalty.withColumn('prev_loyalty',
                           F.lag('is_loyalty').over(window))
df_user_loyalty = df_user_loyalty.filter(F.col('prev_loyalty').isNull() | 
                                         (F.col('prev_loyalty') != F.col('is_loyalty'))
                    ).drop('prev_loyalty')
#df_user_loyalty.display()

In [0]:
df_dim_user_scd2 = (df_user_loyalty
            .withColumn('effective_to_ts',
                           F.lead('creation_time_utc').over(window))
            .withColumnRenamed('creation_time_utc','effective_from_ts')
            .withColumn('is_current',F.col('effective_to_ts').isNull())
            )
df_dim_user_scd2 = df_dim_user_scd2.select('user_id','is_loyalty','effective_from_ts','effective_to_ts','is_current').orderBy('user_id','effective_from_ts')
df_dim_user_scd2.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_user_scd2;

In [0]:
df_dim_user_scd2.write.mode('append').saveAsTable('global_partner_project.gold.dim_user_scd2')

## DIM_USER_CARD

In [0]:
df_dim_user_card = df_order_items.select('user_id','printed_card_number','currency').distinct()
#df_dim_user_card.display()
#print(df_dim_user_card.count()) #23016 rows

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_user_card;

In [0]:
df_dim_user_card.write.mode('append').saveAsTable('global_partner_project.gold.dim_user_card')

## DIM_ITEM

In [0]:
df_dim_item = df_order_items.select('item_name','item_category','restaurant_id').distinct()
#df_dim_item.display()
#print(df_dim_item.count()) #2481 rows

In [0]:
window = (Window.orderBy('restaurant_id','item_category','item_name'))
df_dim_item = df_dim_item.withColumn('item_id',
                                     F.row_number().over(window))
df_dim_item = df_dim_item.select('item_id','item_name','item_category','restaurant_id')
df_dim_item.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_item;

In [0]:
df_dim_item.write.mode('append').saveAsTable('global_partner_project.gold.dim_item')

## DIM_RESTAURANT

In [0]:
df_dim_restaurant = df_order_items.select('restaurant_id','app_name').distinct()
#df_dim_restaurant.display()
#print(df_dim_restaurant.count()) #29 rows

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_restaurant;

In [0]:
df_dim_restaurant.write.mode('append').saveAsTable('global_partner_project.gold.dim_restaurant')

## DIM_OPTIONS

In [0]:
df_order_options = spark.read.table('global_partner_project.silver.order_options')

In [0]:
df_order_options = df_order_options.select('order_id','lineitem_id','option_group_name',
                                           'option_name','option_price','option_quantity').distinct()

In [0]:
#print(df_order_options.count())
#df_order_options.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_options;

In [0]:
df_order_options.write.saveAsTable('global_partner_project.gold.dim_options')

## DIM_DATE

In [0]:
df_dim_date = df_order_items.select('creation_time_utc').distinct()
df_dim_date = (df_dim_date.withColumn('date_key',F.date_format(F.col('creation_time_utc'),'dd-MM-yyyy'))
                       .withColumn('year',F.year(F.col('creation_time_utc')))
                       .withColumn('month',F.month(F.col('creation_time_utc')))
                       .withColumn('week',F.weekofyear(F.col('creation_time_utc')))
                       .withColumn('day_of_week',F.date_format('creation_time_utc','EEEE'))
                       .withColumn('is_weekend',F.when(F.col('day_of_week').isin(['Sunday','Saturday']),True).otherwise(False))
                       .withColumn('holiday_name',F.when(
                           (F.dayofmonth(F.col('creation_time_utc'))==1) &
                           (F.col('month')==1),"New Year's Day")
                            .when((F.dayofmonth(F.col('creation_time_utc'))==2) &
                           (F.col('month')==1),"New Year's Day Observed")
                            .when((F.dayofmonth(F.col('creation_time_utc'))==25) &
                           (F.col('month')==12),"Christmas")
                                   )
                       .withColumn('is_holiday',F.when(F.col('holiday_name').isNotNull(),True).otherwise(False))
                       .withColumn('ingest_date',F.current_timestamp())
)
df_dim_date = df_dim_date.drop('creation_time_utc','')

In [0]:
#df_dim_date = df_dim_date.filter((F.col('holiday_name').isNotNull()))
#df_dim_date.display()

In [0]:
df_date_dim = spark.read.table('global_partner_project.silver.date_dim')

In [0]:
df_date = (df_dim_date.alias('o').join(
                df_date_dim.alias('d'),
                 (
                     (F.col('o.date_key') == F.col('d.date_key'))
                 ), 
                 'outer')
                 .select(
                     F.coalesce(F.col('o.date_key'),F.col('d.date_key')).alias('date_key'),
                     F.coalesce(F.col('o.year'),F.col('d.year')).alias('year'),
                     F.coalesce(F.col('o.month'),F.col('d.month')).alias('month'),
                     F.coalesce(F.col('o.week'),F.col('d.week')).alias('week'),
                     F.coalesce(F.col('o.day_of_week'),F.col('d.day_of_week')).alias('day_of_week'),
                     F.coalesce(F.col('o.is_weekend'),F.col('d.is_weekend')).alias('is_weekend'),
                     F.coalesce(F.col('o.is_holiday'),F.col('d.is_holiday')).alias('is_holiday'),
                     F.coalesce(F.col('o.holiday_name'),F.col('d.holiday_name')).alias('holiday_name'),
                     F.coalesce(F.col('o.ingest_date'),F.col('d.ingest_date')).alias('ingest_date'),
                 )
                 )

'''&
                     (F.col('o.year') == F.col('d.year')) &
                     (F.col('o.month') == F.col('d.month')) &
                     (F.col('o.week') == F.col('d.week'))'''

In [0]:
#df_date.display()

In [0]:
#print(df_date.count())
#agg_df = df_date.groupBy('date_key').agg(F.count('date_key').alias('Cnt')).orderBy(F.col('Cnt').desc())
#agg_df.display()

In [0]:
df_date = df_date.distinct()
#print(df_date.count())
#agg_df = df_date.groupBy('date_key').agg(F.count('date_key').alias('Cnt')).orderBy(F.col('Cnt').desc())
#agg_df.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.dim_date;

In [0]:
df_date.write.mode('append').saveAsTable('global_partner_project.gold.dim_date')

In [0]:
%sql
SELECT * FROM global_partner_project.silver.date_dim

# Creating FACT Table

## FACT_ORDER

In [0]:
df_fact_order = df_order_items.withColumn('date_key',
                                          F.date_format(F.col('creation_time_utc'),'dd-MM-yyyy'))

In [0]:
df_fact_order =  df_fact_order.alias('f').join(
                    df_dim_item.alias('i'),
                   on=['item_name','item_category','restaurant_id'],
                   how='inner')

In [0]:
df_fact_order = df_fact_order.select('order_id','lineitem_id','creation_time_utc','date_key',
                                      'user_id','restaurant_id','item_id',
                                      'item_quantity','item_price','ingest_date').distinct()

In [0]:
#print(df_fact_order.count())
#df_fact_order.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.gold.fact_order;

In [0]:
df_fact_order.write.mode('append').saveAsTable('global_partner_project.gold.fact_order')